### **Setup (only for this file)**
These lines repeat what Parts 1 to 3 already do: the setup, the functions of Part 2, the 12 temporal splits, the candidates of 21.1 and the final set of Part 3 (28.3). They are removed when Part 4 is joined to the main notebook.

In [ ]:
import pandas as pd
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

train_path=os.path.join("..","data","raw","train.csv")
clientes_path=os.path.join("..","data","raw","clientes.csv")

df_train= pd.read_csv(train_path,index_col="ID",sep=",")
df_clientes = pd.read_csv(clientes_path, sep=",")

target_col = "repeat_purchase_90d"
var_groups = {
    "Current transaction (financial)": ["Total a Pagar", "Total Bruto", "Total Liquido", "Total Imp_", "Total Portes",
                                        "Total Desc_ Global", "Total Desc_ Linha", "Custo"],
    "Current transaction (operational)": ["N_ Detalhes", "invoice_count_current_purchase"],
    "Customer historical summary": ["customer_tenure_days", "prior_purchase_count", "prior_total_spend",
                                    "prior_mean_purchase_value", "prior_std_purchase_value",
                                    "prior_min_purchase_value", "prior_max_purchase_value"],
    "Customer cadence & gaps": ["days_since_previous_purchase", "prior_mean_gap_days", "prior_std_gap_days"],
    "Rolling windows (RFM)": ["purchase_count_30d", "spend_30d", "purchase_count_90d", "spend_90d",
                              "purchase_count_180d", "spend_180d", "purchase_count_365d", "spend_365d"],
}

# recover_values of 3.6.7
def recover_values(frame):
    """Fills the cells that can be computed exactly from other columns of the same row (3.6.1 and 3.6.7)."""
    out = frame.copy()
    count, tenure = out["prior_purchase_count"], out["customer_tenure_days"]
    out["prior_mean_purchase_value"] = out["prior_total_spend"] / count.replace(0, np.nan)

    days, gap = out["days_since_previous_purchase"].copy(), out["prior_mean_gap_days"].copy()
    fill = days.isna() & (count == 1)
    out.loc[fill, "days_since_previous_purchase"] = tenure[fill]
    fill = days.isna() & (count >= 2) & gap.notna()
    out.loc[fill, "days_since_previous_purchase"] = (tenure - gap * (count - 1))[fill]
    fill = gap.isna() & (count >= 2) & days.notna()
    out.loc[fill, "prior_mean_gap_days"] = ((tenure - days) / (count - 1))[fill]

    fill = out["spend_365d"].isna() & (out["purchase_count_365d"] == 0)
    out.loc[fill, "spend_365d"] = 0.0
    fill = out["spend_365d"].isna() & (out["purchase_count_365d"] == out["purchase_count_180d"])
    out.loc[fill, "spend_365d"] = out.loc[fill, "spend_180d"]
    fill = out["spend_365d"].isna() & (out["purchase_count_365d"] == count)
    out.loc[fill, "spend_365d"] = out.loc[fill, "prior_total_spend"]

    fill = out["prior_max_purchase_value"].isna() & (count == 1)
    out.loc[fill, "prior_max_purchase_value"] = out.loc[fill, "prior_total_spend"]
    fill = out["prior_max_purchase_value"].isna() & (count == 2)
    out.loc[fill, "prior_max_purchase_value"] = (out["prior_total_spend"] - out["prior_min_purchase_value"])[fill]
    return out

In [ ]:
import warnings
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (OneHotEncoder, OrdinalEncoder, FunctionTransformer,
                                   StandardScaler, MinMaxScaler, RobustScaler)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, f1_score
from sklearn.exceptions import ConvergenceWarning

# without scaling the solver can stop at its limit of iterations. We look at this in section 18, so we hide the warning
warnings.filterwarnings("ignore", category=ConvergenceWarning)

In [ ]:
def signed_log(X):
    """log1p that keeps the sign. For values >= 0 it is the same as log1p, and it also works
    on the negative impossible values of the option "keep as is" (13.4)."""
    return np.sign(X) * np.log1p(np.abs(X))


class Winsorizer(BaseEstimator, TransformerMixin):
    """Clips each column to its 1% and 99% quantiles, learned on the train rows."""
    def fit(self, X, y=None):
        X = np.asarray(X, dtype=float)
        self.low_ = np.nanquantile(X, 0.01, axis=0)
        self.high_ = np.nanquantile(X, 0.99, axis=0)
        return self

    def transform(self, X):
        return np.clip(np.asarray(X, dtype=float), self.low_, self.high_)


def make_preprocessor(log_cols, plain_cols, cat_cols, flag_cols=(), num_fill="median", indicators=False,
                      cat_fill="Missing", outliers="log1p", encoder="onehot", min_frequency=None, scaler="standard"):
    """Layer B: the steps that learn something from the data (medians, levels, scales).
    Inside a Pipeline they are fitted only on the train rows of each split."""
    scalers = {"standard": StandardScaler, "minmax": MinMaxScaler, "robust": RobustScaler}

    def numeric_steps(long_tail):
        if num_fill == "zero":
            steps = [("impute", SimpleImputer(strategy="constant", fill_value=0, add_indicator=indicators))]
        else:
            steps = [("impute", SimpleImputer(strategy=num_fill, add_indicator=indicators))]
        if long_tail and outliers == "log1p":
            steps.append(("log", FunctionTransformer(signed_log)))
        elif long_tail and outliers == "winsorize":
            steps.append(("winsorize", Winsorizer()))
        if scaler in scalers:
            steps.append(("scale", scalers[scaler]()))
        return Pipeline(steps)

    # categories: missing -> "Missing" (or the most frequent level), then encoding
    if cat_fill == "Missing":
        cat_steps = [("impute", SimpleImputer(strategy="constant", fill_value="Missing"))]
    else:
        cat_steps = [("impute", SimpleImputer(strategy="most_frequent"))]
    if encoder == "onehot" and min_frequency is None:
        cat_steps.append(("encode", OneHotEncoder(handle_unknown="ignore", sparse_output=False)))
    elif encoder == "onehot":
        cat_steps.append(("encode", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=min_frequency,
                                                  sparse_output=False)))
    else:
        cat_steps.append(("encode", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)))
        if scaler in scalers:
            cat_steps.append(("scale", scalers[scaler]()))

    parts = [("log", numeric_steps(long_tail=True), list(log_cols)),       # money, counts and days (long tails)
             ("plain", numeric_steps(long_tail=False), list(plain_cols)),  # columns that can be negative, or ratios
             ("cat", Pipeline(cat_steps), list(cat_cols)),
             ("flags", "passthrough", list(flag_cols))]                    # 0/1 columns
    return ColumnTransformer([p for p in parts if len(p[2]) > 0])


def temporal_scores(frame, columns, recipe, row_filter=None, metric="ap"):
    """One score per fold: the instrument is fitted on the train rows and scored on the validation block."""
    used = columns["log_cols"] + columns["plain_cols"] + columns["cat_cols"] + columns["flag_cols"]
    scores = []
    for train_idx, val_idx in folds:
        train, val = frame.loc[train_idx], frame.loc[val_idx]
        if row_filter is not None:
            train = row_filter(train)   # rows are only removed from the train side, never from validation
        pipe = Pipeline([("prep", make_preprocessor(**columns, **recipe)),
                         ("model", LogisticRegression(max_iter=3000))])
        pipe.fit(train[used], train[target_col])
        prob = pipe.predict_proba(val[used])[:, 1]
        if metric == "ap":
            scores.append(average_precision_score(val[target_col], prob))
        elif metric == "f1":
            scores.append(f1_score(val[target_col], prob >= 0.5))
        else:
            scores.append((prob >= 0.5).mean())   # share of rows predicted as 1
    return np.array(scores)


def compare(results, reference):
    """Mean AP of each option and its difference to the reference, fold by fold (paired)."""
    rows = {}
    for name, scores in results.items():
        diff = scores - results[reference]
        sem = diff.std(ddof=1) / np.sqrt(len(diff))
        if name == reference:
            verdict = "reference"
        elif abs(diff.mean()) <= 2 * sem:
            verdict = "inside the noise"
        else:
            verdict = "better" if diff.mean() > 0 else "worse"
        rows[name] = {"AP": round(scores.mean(), 4), "difference": round(diff.mean(), 4), "SEM": round(sem, 4),
                      "better in": f"{(diff > 0).sum()}/{len(diff)}", "verdict": verdict}
    return pd.DataFrame(rows).T


def blank_impossible(frame):
    """Impossible values of 3.6 -> missing. It works row by row, so it learns nothing from the other rows."""
    out = frame.copy()
    count = out["prior_purchase_count"]
    # 3.6.1 to 3.6.3: 1e9, negative values, means bigger than the biggest purchase, spend_365d bigger than the prior spend
    mean = out["prior_mean_purchase_value"]
    out.loc[(mean < 0) | (mean >= 1e8) | (mean > out["prior_max_purchase_value"] + 0.01), "prior_mean_purchase_value"] = np.nan
    spend = out["spend_365d"]
    out.loc[(spend < 0) | (spend >= 1e8) | (spend > out["prior_total_spend"] + 0.01), "spend_365d"] = np.nan
    for col in ["days_since_previous_purchase", "prior_mean_gap_days"]:
        out.loc[out[col] < 0, col] = np.nan
    # 3.6.6: biggest purchase above its limit
    max_limit = out["prior_total_spend"] - (count - 1) * out["prior_min_purchase_value"]
    out.loc[(count >= 1) & (out["prior_max_purchase_value"] > max_limit + 0.01), "prior_max_purchase_value"] = np.nan
    return out


def add_features(frame):
    """New variables, built row by row from the columns of the same purchase."""
    out = frame.copy()
    months = out["customer_tenure_days"] / 30
    out["bought_in_90d"] = (out["purchase_count_90d"] > 0).astype(int)
    out["rhythm_ratio"] = out["days_since_previous_purchase"] / out["prior_mean_gap_days"].replace(0, np.nan)
    out["purchase_rate"] = out["prior_purchase_count"] / (months + 1)
    out["spend_per_month"] = out["prior_total_spend"] / (months + 1)
    out["avg_basket_365d"] = out["spend_365d"] / out["purchase_count_365d"].replace(0, np.nan)
    out["spend_acceleration"] = out["spend_30d"] / (out["spend_90d"] + 1)
    out["margin_ratio"] = (out["Total Liquido"] - out["Custo"]) / (out["Total Liquido"] + 1)
    out["discount_intensity"] = (out["Total Desc_ Linha"] + out["Total Desc_ Global"]) / (out["Total Bruto"] + 1)
    out["value_vs_habit"] = ((out["Total a Pagar"] - out["prior_mean_purchase_value"])
                             / out["prior_std_purchase_value"].replace(0, np.nan))
    return out


def prepare_data(frame, clientes):
    """Layer A: every step that works row by row. The same function will be used on test.csv."""
    out = frame.copy()
    out["purchase_date"] = pd.to_datetime(out["purchase_date"])
    out = blank_impossible(out)
    out = recover_values(out)          # 3.6.7
    out["is_first_purchase"] = (out["prior_purchase_count"] == 0).astype(int)
    out = out.join(clientes.set_index("customer_id")[["country_or_region", "commercial_region", "island",
                                                      "municipality", "postal_area"]],
                   on="customer_id", validate="many_to_one")
    out["incomplete_address"] = (out["municipality"].isna() & out["postal_area"].isna()).astype(int)
    out = add_features(out)
    return out

In [ ]:
# 12.1 and 12.2: rows without the empty ones, development rows and the 12 temporal splits
empty_rows = df_train.drop(columns=["customer_id", target_col]).isna().all(axis=1)
df_model = prepare_data(df_train.loc[~empty_rows], df_clientes)
is_dev = df_model["purchase_date"] < "2024-01-01"
dev = df_model.loc[is_dev]
block_starts = pd.date_range("2018-01-01", "2023-07-01", freq="6MS")
folds = []
for start in block_starts:
    end = start + pd.DateOffset(months=6)
    train_idx = dev.index[dev["purchase_date"] < start - pd.Timedelta(days=90)]
    val_idx = dev.index[(dev["purchase_date"] >= start) & (dev["purchase_date"] < end)]
    folds.append((train_idx, val_idx))

# 21.1: the candidates and the recipe that Part 2 hands to Part 3
final_columns = {
    "log_cols": [
        "Total a Pagar", "Total Bruto", "Total Liquido", "Total Imp_", "Total Portes",
        "Total Desc_ Global", "Total Desc_ Linha", "Custo", "N_ Detalhes",
        "invoice_count_current_purchase", "customer_tenure_days", "prior_purchase_count",
        "prior_total_spend", "prior_mean_purchase_value", "prior_std_purchase_value",
        "prior_min_purchase_value", "prior_max_purchase_value", "days_since_previous_purchase",
        "prior_mean_gap_days", "prior_std_gap_days", "purchase_count_30d", "spend_30d",
        "purchase_count_90d", "spend_90d", "purchase_count_180d", "spend_180d", "purchase_count_365d",
        "spend_365d", "rhythm_ratio", "purchase_rate", "spend_per_month", "avg_basket_365d"],
    "plain_cols": [
        "random_noise", "spend_acceleration", "margin_ratio", "discount_intensity", "value_vs_habit"],
    "cat_cols": [
        "payment_type", "salesperson", "commercial_zone", "transport_method", "warehouse",
        "document_series", "country_or_region", "commercial_region", "island"],
    "flag_cols": [
        "is_first_purchase", "incomplete_address"],
}
final_recipe = {"num_fill": "median", "indicators": False, "cat_fill": "Missing", "outliers": "log1p", "encoder": "onehot", "min_frequency": 10, "scaler": "standard"}
feature_cols = final_columns["log_cols"] + final_columns["plain_cols"] + final_columns["cat_cols"] + final_columns["flag_cols"]
print(f"Rows: {len(df_model):,} | Development: {is_dev.sum():,} | Candidates: {len(feature_cols)}")

In [ ]:
# 28.3: the final set of Part 3
selected = ["Total Bruto", "prior_max_purchase_value", "prior_std_gap_days", "purchase_count_90d", "purchase_count_365d",
            "purchase_rate", "avg_basket_365d", "margin_ratio", "discount_intensity", "salesperson", "transport_method",
            "is_first_purchase"]
selected_columns = {group: [c for c in cols if c in selected] for group, cols in final_columns.items()}
print(f"Final set of Part 3: {len(selected)} variables")

### **29. Modelling: The Plan**

Part 4 builds the model that predicts `repeat_purchase_90d` for test.csv and evaluates it with the F1 of class 1, the metric of the competition. We compare **six algorithms** that learn in different ways: logistic regression, gradient boosting (`HistGradientBoostingClassifier`), Naive Bayes, KNN, a decision tree and a random forest.

The plan follows the rules of the previous parts:
* every option is fitted on the train rows of the same 12 temporal splits and scored on the validation blocks (12.2);
* the algorithms and their settings are compared with the AP, as in Parts 2 and 3, because it does not depend on the threshold;
* then the threshold is chosen for the F1 (2.3), with the predictions of the 12 blocks;
* the holdout (2024 and 2025), never used until now, gives the honest check of the chosen model (34);
* the final model learns from all the rows of train.csv and predicts test.csv (35).

`split_probs` gives the probabilities of each validation block, and `board` puts the options side by side: the AP with `compare` (paired, with the SEM) and the best F1 that each option can reach.

In [ ]:
from sklearn.base import clone
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import confusion_matrix


def split_probs(model, columns):
    """The probabilities of each validation block, from the pipeline fitted on the train rows of its split."""
    probs = []
    for train_idx, val_idx in folds:
        pipe = Pipeline([("prep", make_preprocessor(**columns, **final_recipe)), ("model", clone(model))])
        pipe.fit(df_model.loc[train_idx, feature_cols], df_model.loc[train_idx, target_col])
        probs.append(pipe.predict_proba(df_model.loc[val_idx, feature_cols])[:, 1])
    return probs

# True target of each validation block, and the thresholds we test for the F1.
y_blocks = [df_model.loc[val_idx, target_col].to_numpy() for _, val_idx in folds]
thresholds = np.round(np.arange(0.05, 0.96, 0.01), 2)


def block_ap(probs):
    """AP of each validation block, as in Parts 2 and 3."""
    return np.array([average_precision_score(y, p) for y, p in zip(y_blocks, probs)])


def f1_curve(probs):
    """F1 of class 1 of the 12 blocks together, for each threshold."""
    y_all, p_all = np.concatenate(y_blocks), np.concatenate(probs)
    return pd.Series([f1_score(y_all, p_all >= t) for t in thresholds], index=thresholds)


def board(probs_by_name, reference):
    """compare (AP, paired) plus the best F1 that each option can reach with its own threshold."""
    table = compare({name: block_ap(p) for name, p in probs_by_name.items()}, reference)
    table["best F1"] = [round(f1_curve(p).max(), 4) for p in probs_by_name.values()]
    return table

#F1 of always predicting 1 (the minimum any model has to beat).
floor_f1 = f1_score(np.concatenate(y_blocks), np.ones(sum(len(y) for y in y_blocks)))
print(f"F1 of always predicting 1 in the 12 blocks: {floor_f1:.3f}")

Always predicting 1 already gives an F1 of 0.706 on the 12 blocks (2.3). Any model has to beat this.

### **30. Candidate Algorithms with Default Settings**

Each algorithm looks at the data in a different way. Logistic regression follows the monotonic relations of the EDA (section 7), gradient boosting and the decision tree can find breaks like the 90 days of 7.3, the random forest averages many trees, KNN compares each purchase with the most similar ones, and Naive Bayes is a simple probabilistic reference.

We run the six with their default settings, each with the final set of Part 3 (28.3) and with all the 48 candidates. Part 3 chose the variables with a linear model, so testing both sets lets the data show which one suits each algorithm.

In [ ]:
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# The six algorithms with their default settings (random_state=44 where the algorithm uses randomness).
models = {"logistic regression": LogisticRegression(max_iter=3000),
          "gradient boosting": HistGradientBoostingClassifier(random_state=44),
          "naive bayes": GaussianNB(),
          "knn": KNeighborsClassifier(),
          "decision tree": DecisionTreeClassifier(random_state=44),
          "random forest": RandomForestClassifier(random_state=44, n_jobs=-1)}
variable_sets = {"final set": selected_columns, "all candidates": final_columns}

probs = {f"{m}, {s}": split_probs(model, columns) for m, model in models.items() for s, columns in variable_sets.items()}
board(probs, "logistic regression, final set")

With the default settings the logistic regression with the 12 selected variables is the best, and every other algorithm is clearly **worse**. Gradient boosting and the random forest are close to each other (AP about 0.82), KNN and Naive Bayes are far behind (0.73 to 0.75) and the decision tree is the weakest (about 0.60), because without limits it keeps splitting until it learns the train rows by heart. Naive Bayes with the final set even reaches a best F1 of only 0.59, below always predicting 1 (0.706). Each algorithm is tuned in 31 before the real comparison.

### **31. Tuning**

We tune each algorithm on the same 12 splits, using the AP, with small grids: every extra setting tried on the same blocks makes the best result a bit more optimistic. Logistic regression and gradient boosting are tuned with simple loops (31.1 and 31.2). For the four new algorithms we use `GridSearchCV`, and `RandomizedSearchCV` for the random forest, with our 12 temporal splits as `cv`, so every combination is scored on exactly the same blocks. Each search runs on both variable sets, and when the best value is at the edge of a grid we widen it.

In [ ]:
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV

# The development rows, in the same order as df_model.
Xdev = df_model.loc[is_dev, feature_cols]
ydev = df_model.loc[is_dev, target_col]
# folds holds ID labels, but the search needs positions inside Xdev.
cv_folds = [(Xdev.index.get_indexer(tr), Xdev.index.get_indexer(va)) for tr, va in folds]


def tune(model, grid, columns, random=False, n_iter=20):
    """Searches the grid on the 12 temporal splits, with the same Layer B and the AP."""
    pipe = Pipeline([("prep", make_preprocessor(**columns, **final_recipe)), ("model", model)])
    grid = {f"model__{k}": v for k, v in grid.items()}
    if random:
        search = RandomizedSearchCV(pipe, grid, n_iter=n_iter, scoring="average_precision", cv=cv_folds,
                                    random_state=44, refit=False)
    else:
        search = GridSearchCV(pipe, grid, scoring="average_precision", cv=cv_folds, refit=False)
    return search.fit(Xdev, ydev)


def best_of(searches):
    """The variable set and the settings with the best AP, from one search per variable set."""
    best_set = max(searches, key=lambda s: searches[s].best_score_)
    settings = {k.replace("model__", ""): v for k, v in searches[best_set].best_params_.items()}
    return best_set, settings


def show_search(searches, top=5):
    """Best AP of each variable set, and the best combinations of the winning set."""
    for s, search in searches.items():
        print(f"{s}: best AP {search.best_score_:.4f} with {search.best_params_}")
    table = pd.DataFrame(searches[best_of(searches)[0]].cv_results_)
    table = table[[c for c in table.columns if c.startswith("param_")] + ["mean_test_score"]]
    table.columns = [c.replace("param_model__", "") for c in table.columns[:-1]] + ["AP"]
    return table.sort_values("AP", ascending=False).head(top).round(4)


# Each tuned algorithm with its variable set, and its probabilities in the 12 blocks (compared in 32).
tuned_models, tuned_probs = {}, {}


def keep_tuned(name, model, set_name):
    """Stores the tuned version of an algorithm and its probabilities in the 12 blocks."""
    tuned_models[name] = (model, set_name)
    tuned_probs[name] = split_probs(model, variable_sets[set_name])

#### **31.1. Logistic Regression**

In [ ]:
C_grid = [0.01, 0.03, 0.1, 0.3, 1, 3, np.inf] #C = infinity means no penalty at all. It shows if the penalty is worth it.
lr_probs = {C: split_probs(LogisticRegression(C=C, max_iter=3000), selected_columns) for C in C_grid}
lr_table = pd.DataFrame({"AP": [block_ap(p).mean() for p in lr_probs.values()],
                         "best F1": [f1_curve(p).max() for p in lr_probs.values()]}, index=pd.Index(C_grid, name="C")).round(4)
best_C = float(lr_table["AP"].idxmax())
print(f"Best C: {best_C}")
lr_table

The last row is the model with no penalty (C = infinity), to check whether the penalty helps. It is clearly the worst (AP of about 0.82 against about 0.84), so it does. Between the other values the AP barely changes, and the best is C = 0.1, a slightly stronger penalty than the default (C = 1).

#### **31.2. Gradient Boosting**

In [ ]:
#5 settings, from the default (big trees that learn fast) to small trees that learn slowly.
gb_grid = [dict(learning_rate=0.1, max_leaf_nodes=31, min_samples_leaf=20, max_iter=100),
           dict(learning_rate=0.05, max_leaf_nodes=15, min_samples_leaf=40, max_iter=200),
           dict(learning_rate=0.03, max_leaf_nodes=7, min_samples_leaf=80, max_iter=300),
           dict(learning_rate=0.03, max_leaf_nodes=4, min_samples_leaf=80, max_iter=400),
           dict(learning_rate=0.05, max_leaf_nodes=4, min_samples_leaf=150, max_iter=200)] 
gb_probs, rows = {}, []
for i, settings in enumerate(gb_grid):
    model = HistGradientBoostingClassifier(**settings, l2_regularization=1.0, random_state=44)
    row = {"leaves": settings["max_leaf_nodes"], "learning rate": settings["learning_rate"],
           "min rows per leaf": settings["min_samples_leaf"], "trees": settings["max_iter"]}
    for s, columns in variable_sets.items():
        gb_probs[(i, s)] = split_probs(model, columns)
        row[f"AP, {s}"] = round(block_ap(gb_probs[(i, s)]).mean(), 4)
        row[f"best F1, {s}"] = round(f1_curve(gb_probs[(i, s)]).max(), 4)
    rows.append(row)
best_gb = max(gb_probs, key=lambda k: block_ap(gb_probs[k]).mean())
print(f"Best gradient boosting: settings {best_gb[0] + 1} with the {best_gb[1]}")
pd.DataFrame(rows, index=pd.Index(range(1, len(gb_grid) + 1), name="settings"))

Smaller trees that learn slowly work much better. Going from the default (settings 1) to trees with 4 leaves, a learning rate of 0.03 and at least 80 rows per leaf (settings 4) raises the AP from about 0.82 to about 0.84. A tree with 4 leaves only combines two or three variables, so the best boosting ends up close to an additive model, like the logistic regression. This matches the EDA, where the strong signals are monotonic and add up. With small trees, boosting does slightly better with all 48 candidates, so we keep them.

#### **31.3. Naive Bayes**

In [ ]:
nb_grid = {"var_smoothing": np.logspace(-9, 3, 13)}  # from 1e-9 (the default) to 1000
nb_search = {s: tune(GaussianNB(), nb_grid, columns) for s, columns in variable_sets.items()}
show_search(nb_search)

The best AP comes with a very large `var_smoothing` (10 or more), and the final set is clearly better than the 48 candidates (0.8375 against 0.8255). The search picks 100, but from 10 to 1000 the AP is the same, so the next table shows what the smoothing does to the probabilities.

In [ ]:
# What the smoothing does to the probabilities, with the final set.
nb_rows = []
for v in [1e-9, 0.01, 0.1, 1, 10, 100]:
    p = split_probs(GaussianNB(var_smoothing=v), selected_columns)
    p_all = np.concatenate(p)
    nb_rows.append({"var_smoothing": v, "AP": block_ap(p).mean(), "best F1": f1_curve(p).max(),
                    "lowest probability": p_all.min(), "highest probability": p_all.max(),
                    "share below 0.05 or above 0.95": ((p_all < 0.05) | (p_all > 0.95)).mean()})
pd.DataFrame(nb_rows).set_index("var_smoothing").round(4)

With the default smoothing, 89% of the probabilities are below 0.05 or above 0.95, and the best F1 (0.59) is below always predicting 1. Our variables are strongly correlated (section 8) and Naive Bayes treats them as independent, so it counts the same signal several times and becomes overconfident. More smoothing pulls the probabilities toward the middle and raises the AP to 0.8375, but at 100 they all fall between 0.53 and 0.58, and the best F1 (0.707) is no better than always predicting 1 (0.706).

We keep `var_smoothing = 10`: the same AP as 100, with probabilities that still spread from 0.38 to 0.77.

In [ ]:
keep_tuned("naive bayes", GaussianNB(var_smoothing=10), "final set")

#### **31.4. KNN**

In [ ]:
knn_grid = {"n_neighbors": [5, 15, 25, 51, 101, 151], "weights": ["uniform", "distance"], "p": [1, 2]}
knn_search = {s: tune(KNeighborsClassifier(), knn_grid, columns) for s, columns in variable_sets.items()}
knn_set, knn_settings = best_of(knn_search)
keep_tuned("knn", KNeighborsClassifier(**knn_settings), knn_set)
show_search(knn_search)

KNN goes from an AP of 0.752 with the default settings to 0.840, the same level as the logistic regression. It needs many neighbours (101): with only 5, each prediction depends on a handful of purchases and is noisy. The best combinations all give more weight to the closest neighbours (`distance`), the Manhattan distance (`p = 1`) beats the Euclidean for the same number of neighbours, and the final set beats the 48 candidates (0.8400 against 0.8376).

#### **31.5. Decision Tree**

In [ ]:
dt_grid = {"max_depth": [5, 8, None], "min_samples_leaf": [20, 40, 80, 150], "min_samples_split": [2, 50],
           "ccp_alpha": [0.0, 0.0001, 0.001]}
dt_search = {s: tune(DecisionTreeClassifier(random_state=44), dt_grid, columns) for s, columns in variable_sets.items()}
dt_set, dt_settings = best_of(dt_search)
keep_tuned("decision tree", DecisionTreeClassifier(**dt_settings, random_state=44), dt_set)
show_search(dt_search)

In [ ]:
# AP of the winning set by leaf size and pruning, with no depth limit and min_samples_split = 2.
dt_results = pd.DataFrame(dt_search[dt_set].cv_results_)
no_limit = dt_results["param_model__max_depth"].isna() & (dt_results["param_model__min_samples_split"] == 2)
(dt_results[no_limit].pivot_table(index="param_model__min_samples_leaf", columns="param_model__ccp_alpha",
                                  values="mean_test_score")
 .rename_axis(index="min_samples_leaf", columns="ccp_alpha").round(4))

For the tree, the leaf size is what matters. With at least 80 rows per leaf and the final set, the AP goes from 0.615 to 0.811, and the depth limit and `min_samples_split` change almost nothing (the five best combinations have the same AP). The table shows that pruning with `ccp_alpha` does the same job as the leaf size: it helps when the leaves are small (0.7775 to 0.7964 with 20 rows) and lowers the AP when they already hold 80 rows (0.8112 to 0.8061). The search keeps a tiny `ccp_alpha` of 0.0001, which gives the same AP as no pruning.

The tree is still the weakest algorithm, since it gives the same probability to every purchase in a leaf.

#### **31.6. Random Forest**

In 26.2 the random forest was only used to rank the variables. Here it is a model, tuned with a random search of 30 of the 180 combinations on both sets.

In [ ]:
# Random search of 30 of the 180 combinations, on both sets. It takes about 15 minutes, so it is commented.
# Result: final set, n_estimators=300, max_depth=6, min_samples_leaf=5, max_features="sqrt", AP 0.8413 (all candidates: 0.8401).
# rf_grid = {"n_estimators": [200, 300, 500], "max_depth": [None, 6, 8, 12], "min_samples_leaf": [5, 10, 20, 40, 80],
#            "max_features": ["sqrt", 0.3, 0.5]}
# rf_search = {s: tune(RandomForestClassifier(random_state=44, n_jobs=-1), rf_grid, columns, random=True, n_iter=30)
#              for s, columns in variable_sets.items()}
# show_search(rf_search)

In [ ]:
# Three settings of the search are at the edge of their grid: min_samples_leaf = 5 (the smallest), max_depth = 6
# (the smallest limit) and max_features = "sqrt" (the fewest columns). We move each one past the edge, with the others fixed.
rf_best = dict(n_estimators=300, max_depth=6, min_samples_leaf=5, max_features="sqrt", random_state=44, n_jobs=-1)
rf_edges = {"min_samples_leaf": [1, 2], "max_depth": [4, 5], "max_features": ["log2"]}
rf_probs = {"search result": split_probs(RandomForestClassifier(**rf_best), selected_columns)}
for setting, values in rf_edges.items():
    for v in values:
        rf_probs[f"{setting} = {v}"] = split_probs(RandomForestClassifier(**{**rf_best, setting: v}), selected_columns)

tuned_models["random forest"] = (RandomForestClassifier(**rf_best), "final set")
tuned_probs["random forest"] = rf_probs["search result"]
board(rf_probs, "search result")

The best forest uses the final set, 300 trees, a depth of at most 6, at least 5 rows per leaf and `sqrt` features per split, with an AP of 0.8413 (0.8401 with the 48 candidates). Three of these settings are at the edge of their grid, so we moved each one past it: smaller leaves, shallower trees and fewer columns per split (`log2`) give APs from 0.8398 to 0.8417, all inside the noise. We keep the result of the search, and as with gradient boosting, shallow trees work best.

### **32. Choosing the Model**

Now the tuned version of each algorithm side by side, plus the average of the probabilities of the logistic regression and gradient boosting. These two look at the data in different ways (a linear model on 12 variables and small trees on 48), so they make different mistakes and their average can beat both.

In [ ]:
# The tuned logistic regression (31.1) and gradient boosting (31.2) join the four algorithms of 31.3 to 31.6.
tuned_models["logistic regression"] = (LogisticRegression(C=best_C, max_iter=3000), "final set")
tuned_probs["logistic regression"] = lr_probs[best_C]
tuned_models["gradient boosting"] = (HistGradientBoostingClassifier(**gb_grid[best_gb[0]], l2_regularization=1.0,
                                                                    random_state=44), best_gb[1])
tuned_probs["gradient boosting"] = gb_probs[best_gb]

finalists = {name: tuned_probs[name] for name in models}  # the tuned version of each algorithm, in the order of 30
finalists["average of the two"] = [(a + b) / 2 for a, b in zip(lr_probs[best_C], gb_probs[best_gb])]
board(finalists, "logistic regression")

Gradient boosting, Naive Bayes, KNN and the random forest are all **inside the noise** of the logistic regression (AP between 0.8375 and 0.8413), and only the decision tree is clearly worse. The random forest has the highest AP of a single algorithm, but it beats the logistic regression in only 7 of 12 blocks. The average of logistic regression and gradient boosting is still the only option **clearly ahead** (better in 11 of 12 blocks) and has the best F1.

In [ ]:
def train_ap(model, columns):
    """Mean AP on the train rows of the 12 splits. Far above the validation AP means the model learned its train rows by heart."""
    scores = []
    for train_idx, _ in folds:
        pipe = Pipeline([("prep", make_preprocessor(**columns, **final_recipe)), ("model", clone(model))])
        pipe.fit(df_model.loc[train_idx, feature_cols], df_model.loc[train_idx, target_col])
        scores.append(average_precision_score(df_model.loc[train_idx, target_col],
                                              pipe.predict_proba(df_model.loc[train_idx, feature_cols])[:, 1]))
    return np.mean(scores)


# Each algorithm before and after tuning, both with the variable set that the tuning chose.
before_after = {}
for name in models:
    tuned, set_name = tuned_models[name]
    columns, default_probs = variable_sets[set_name], probs[f"{name}, {set_name}"]
    before_after[name] = {"variables": set_name,
                          "train AP, default": train_ap(models[name], columns),
                          "AP, default": block_ap(default_probs).mean(),
                          "train AP, tuned": train_ap(tuned, columns),
                          "AP, tuned": block_ap(tuned_probs[name]).mean(),
                          "best F1, default": f1_curve(default_probs).max(),
                          "best F1, tuned": f1_curve(tuned_probs[name]).max()}
pd.DataFrame.from_dict(before_after, orient="index").round(4)

Tuning matters most for the new algorithms: the AP rises by about 0.02 for the random forest, 0.09 for KNN, 0.11 for Naive Bayes and 0.20 for the decision tree, while the logistic regression barely moves (+0.003).

The train AP shows why. The default tree and forest reach 1.0000 on their train rows, so they learned them by heart, and tuning brings them down to 0.83 and 0.87. KNN with `distance` weights still scores 1.0000 on its train rows, because every row is its own closest neighbour, but its validation AP is as good as the logistic regression's, so here this does not hurt.

In [ ]:
# Does it help to add the random forest or KNN to the average? The average of the two is the reference.
averages = {"average of the two": ["logistic regression", "gradient boosting"],
            "+ random forest": ["logistic regression", "gradient boosting", "random forest"],
            "+ knn": ["logistic regression", "gradient boosting", "knn"],
            "+ random forest + knn": ["logistic regression", "gradient boosting", "random forest", "knn"]}
average_probs = {name: [np.mean(blocks, axis=0) for blocks in zip(*[tuned_probs[m] for m in members])]
                 for name, members in averages.items()}
board(average_probs, "average of the two")

Adding the random forest, KNN or both to the average changes the AP by less than 0.001, always inside the noise, and the best F1 stays between 0.747 and 0.749. So the final model stays the **average of the two**: the logistic regression with the 12 selected variables and gradient boosting with small trees on all 48 candidates. None of the new algorithms changes the choice, so the Kaggle submission stays the same.

In [ ]:
# The final model: the two finalists (each with its own variables) and the average of their probabilities.
finalist_models = {"logistic regression": (LogisticRegression(C=best_C, max_iter=3000), selected_columns),
                   "gradient boosting": (HistGradientBoostingClassifier(**gb_grid[best_gb[0]], l2_regularization=1.0,
                                                                        random_state=44), variable_sets[best_gb[1]])}


def fit_average(rows):
    """Fits the pipeline of each finalist on the given rows."""
    pipes = []
    for model, columns in finalist_models.values():
        pipe = Pipeline([("prep", make_preprocessor(**columns, **final_recipe)), ("model", clone(model))])
        pipes.append(pipe.fit(df_model.loc[rows, feature_cols], df_model.loc[rows, target_col]))
    return pipes


def predict_average(pipes, frame):
    """The mean of the probabilities of the two pipelines."""
    return np.mean([pipe.predict_proba(frame[feature_cols])[:, 1] for pipe in pipes], axis=0)


final_probs = finalists["average of the two"]
for name, (model, columns) in finalist_models.items():
    print(f"{name}: {model} | {sum(len(c) for c in columns.values())} variables")

### **33. The Threshold**

The F1 needs a threshold: predict 1 when the probability is above it. The default is 0.5, but nothing says that 0.5 is the best for the F1 (2.3). We choose it with the predictions of the 12 validation blocks together, so each prediction comes from a model that did not see its purchase.

In [ ]:
curve = f1_curve(final_probs)
threshold = curve.idxmax()
near_best = curve[curve >= curve.max() - 0.005].index #Thresholds where the F1 is less than 0.005 from the best (the shaded zone of the plot).

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(curve.index, curve.values, color="#1779A8", label="F1 of the 12 validation blocks")
ax.axhline(floor_f1, color="grey", linestyle="--", linewidth=1, label="always predicting 1")
ax.axvline(threshold, color="#C96A00", linewidth=1.5, label=f"best threshold ({threshold})")
ax.axvline(0.5, color="black", linestyle=":", linewidth=1, label="default threshold (0.5)")
ax.axvspan(near_best.min(), near_best.max(), color="#C96A00", alpha=0.12, label="less than 0.005 from the best F1")
ax.set(xlabel="threshold (predict 1 when the probability is above it)", ylabel="F1 of class 1",
       title="F1 against the threshold, average of the two models")
ax.set_ylim(floor_f1 - 0.05, curve.max() + 0.02)
ax.legend(fontsize=8, loc="lower left")
plt.tight_layout()
plt.show()

#F1 of each block: at our threshold, at 0.5 and when always predicting 1.
per_block = pd.DataFrame({"F1 at the threshold": [f1_score(y, p >= threshold) for y, p in zip(y_blocks, final_probs)],
                          "F1 at 0.5": [f1_score(y, p >= 0.5) for y, p in zip(y_blocks, final_probs)],
                          "F1 always 1": [f1_score(y, np.ones(len(y))) for y in y_blocks]})
print(f"Threshold: {threshold} | F1: {curve.max():.3f} (at 0.5: {curve[0.5]:.3f}, always 1: {floor_f1:.3f})")
print(f"Thresholds within 0.005 of the best F1: {near_best.min()} to {near_best.max()}")
print(f"Blocks where the model beats always predicting 1: {(per_block['F1 at the threshold'] > per_block['F1 always 1']).sum()}/12")
per_block.round(3).T

The best threshold is clearly below 0.5. For the F1 of class 1, missing a repeat purchase costs more than predicting one that does not happen, so it pays to predict 1 a bit more often. The curve is flat around the best point (between about 0.35 and 0.46 the F1 changes by less than 0.005), so the exact value matters little and can move slightly from one computer to another, but 0.5 is already outside this zone.

With this threshold the model reaches an F1 of about 0.75 against 0.706 for always predicting 1, and it beats it in 11 of the 12 blocks. The only exception is a block where always predicting 1 is already very strong (0.756).

### **34. Holdout Check**

The holdout (2024 and 2025, section 12.1) was not used for any decision in Parts 2 to 4, so it gives an honest test of the whole pipeline, on purchases closer to the test period. Both models are trained on 2015 to 2023 only, with the threshold of section 33.

In [ ]:
dev_rows, holdout_rows = df_model.index[is_dev], df_model.index[~is_dev]
pipes = fit_average(dev_rows) #Both models learn only from the development rows (2015-2023)
p_holdout = predict_average(pipes, df_model.loc[holdout_rows])
y_holdout = df_model.loc[holdout_rows, target_col].to_numpy()

print(f"Holdout: {len(holdout_rows)} purchases (2024-2025) | repeat rate: {y_holdout.mean():.3f}")
print(f"AP: {average_precision_score(y_holdout, p_holdout):.3f}")
print(f"F1 at the threshold ({threshold}): {f1_score(y_holdout, p_holdout >= threshold):.3f}"
      f" | at 0.5: {f1_score(y_holdout, p_holdout >= 0.5):.3f} | always 1: {f1_score(y_holdout, np.ones(len(y_holdout))):.3f}")
print(f"Purchases predicted as 1: {(p_holdout >= threshold).mean():.0%}")
pd.DataFrame(confusion_matrix(y_holdout, p_holdout >= threshold),
             index=["no repeat (0)", "repeat (1)"], columns=["predicted 0", "predicted 1"])

On the holdout the model reaches an **F1 of about 0.79** (AP 0.864), against 0.754 for always predicting 1. The gain over this floor (about +0.04) is similar to the one on the 12 blocks, so our choices were not too optimistic. The chosen threshold also beats 0.5 here.

The values are higher than in the 12 blocks because the repeat rate of the holdout is higher (0.605): when more purchases come back, the F1 of class 1 is easier, also for the floor. The confusion matrix shows the cost of the threshold: the model finds about 90% of the repeat purchases, and in exchange more than half of the purchases that do not come back are predicted as 1.

### **35. Final Model and Submission**

The final model is the same average of the two models, now fitted on all the 6,469 purchases of train.csv, including the holdout. The most recent purchases are the closest to test.csv, so the model should learn from them too. Then test.csv follows the path of 21.4. It goes through `prepare_data` (Layer A, row by row), and the fitted pipelines only transform it, they never learn from it. The threshold is the one of section 33.

In [ ]:
# the final model learns from all the 6,469 purchases of train.csv: development rows and holdout
final_pipes = fit_average(df_model.index)

# test.csv goes through the same Layer A, and the pipeline only transforms it (never fit)
test_path = os.path.join("..", "data", "raw", "test.csv")
df_test = pd.read_csv(test_path, index_col="ID", sep=",")
test_model = prepare_data(df_test, df_clientes)
p_test = predict_average(final_pipes, test_model)

submission = pd.DataFrame({"ID": test_model.index, target_col: (p_test >= threshold).astype(int)})
sample = pd.read_csv(os.path.join("..", "data", "raw", "sample_submission.csv"))
print(f"Test purchases: {len(submission):,} | same IDs and order as sample_submission.csv: {submission['ID'].equals(sample['ID'])}")
print(f"Purchases predicted as 1: {submission[target_col].mean():.0%} (holdout: {(p_holdout >= threshold).mean():.0%})")

os.makedirs(os.path.join("..", "data", "processed"), exist_ok=True)
submission.to_csv(os.path.join("..", "data", "processed", "submission_group44.csv"), index=False)
submission.head()

test.csv has 986 purchases, with the same IDs and order as `sample_submission.csv`. The share predicted as 1 is close to the one of the holdout, so test.csv looks similar to the latest purchases of train.csv. The file is saved as `submission_group44.csv` in `data/processed`.

In [ ]:
# Coefficients of the logistic regression of the final model (the columns are scaled, so they can be compared).
lr_pipe = final_pipes[0]
prep = lr_pipe.named_steps["prep"]
names = []
for name, steps, cols in prep.transformers_:
    if name == "cat":
        names += list(steps.named_steps["encode"].get_feature_names_out(cols))
    elif name != "remainder":
        names += list(cols)
coefs = pd.Series(lr_pipe.named_steps["model"].coef_[0], index=names)
numeric = coefs[[n for n in names if n in selected]].sort_values() #We only plot the numeric variables and the flag (the category levels would be too many bars).

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh(numeric.index, numeric.values, color=np.where(numeric > 0, "#1779A8", "#C96A00"))
ax.axvline(0, color="black", linewidth=0.8)
ax.set(xlabel="coefficient (blue: more chance of a repeat purchase, orange: less)",
       title="Logistic regression of the final model: numeric variables and flag")
plt.tight_layout()
plt.show()

The coefficients show what the logistic regression learned (the numeric ones can be compared because the columns are scaled). Frequency is the strongest signal, with `purchase_count_365d`, `purchase_rate` and `purchase_count_90d` at the top. Clients who buy often, and recently, are more likely to come back. Discounts, invoice value and margin add a little. `prior_std_gap_days` is the only negative one, so clients with irregular gaps between purchases come back less.

`is_first_purchase` has a large positive coefficient, which looks odd after 7.3 (first purchases come back less). But the flag never acts alone. In a first purchase all the counts are 0, which pushes the probability down a lot, and the flag only offsets part of that. Overall, first purchases still get a low probability.

### **36. Summary of Part 4**

| Section | Decision | Options tested | Choice | Why |
|---|---|---|---|---|
| 29 | Algorithms | – | Logistic regression, gradient boosting, Naive Bayes, KNN, decision tree, random forest | Six algorithms that learn in different ways |
| 30 | Variables | Final set of Part 3, all the 48 candidates | Final set for every algorithm except gradient boosting | Best AP of each algorithm after tuning |
| 31.1 | Penalty of the logistic regression | C from 0.01 to 3, and no penalty (C = infinity) | C = 0.1 | Best AP; without penalty the AP is clearly lower |
| 31.2 | Settings of gradient boosting | 5 settings, from the default to small trees | 4 leaves, learning rate 0.03, 80 rows per leaf, 400 trees | The default overfits; small trees are clearly better |
| 31.3 | Smoothing of Naive Bayes | `var_smoothing` from 1e-9 to 1000 | `var_smoothing` = 10 | Same AP as larger values, and the probabilities still spread |
| 31.4 | Settings of KNN | 6 numbers of neighbours, 2 weightings, 2 distances | 101 neighbours, `distance` weights, Manhattan distance | Best AP |
| 31.5 | Settings of the decision tree | Depth, rows per leaf, `min_samples_split`, `ccp_alpha` | 80 rows per leaf, no depth limit, almost no pruning | Best AP; pruning adds nothing once the leaves are large |
| 31.6 | Settings of the random forest | 30 random combinations of 180, then past the edges of the grid | 300 trees, depth 6, 5 rows per leaf, `sqrt` features | Best AP of the search; past the edges the AP stays inside the noise |
| 32 | Final model | Six tuned algorithms, average of logistic regression and gradient boosting, averages with the random forest and KNN | Average of logistic regression and gradient boosting | The only option outside the noise; adding the forest or KNN changes nothing |
| 33 | Threshold | 0.05 to 0.95 | Best F1 of the 12 blocks (below 0.5) | The curve is flat around it; 0.5 is outside the best zone |
| 34 | Holdout check | – | F1 about 0.79 against 0.754 for always 1 | The gain is close to the one of the 12 blocks |
| 35 | Submission | – | Fitted on all train.csv, `submission_group44.csv` | The most recent purchases are the closest to test.csv |